In [ ]:
import os
import json
import glob
import warnings
import numpy as np
import xarray as xr
import proplot as pplt
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
RAWDIR     = CONFIGS['filepaths']['raw']
INTERIMDIR = CONFIGS['filepaths']['interim']
SPLITSDIR  = CONFIGS['filepaths']['splits']
WEIGHTSDIR = CONFIGS['filepaths']['weights']
FIELDVARS  = CONFIGS['experiments']['sr']['runs']['sr_atm']['fieldvars']
SEEDS      = CONFIGS['experiments']['nn']['seeds']
LATRANGE   = CONFIGS['domain']['latrange']
LONRANGE   = CONFIGS['domain']['lonrange']

In [ ]:
interimfiles = sorted(glob.glob(os.path.join(INTERIMDIR,'*.nc')))
print(f'Found {len(interimfiles)} files in interim/\n')
print(f'{"Variable":>14s}  {"Shape":>30s}  {"NaN count":>12s}  {"NaN %":>8s}  {"Dims"}')
print('-'*90)

NANVARS = {}
ALLVARS = {}
for filepath in interimfiles:
    da = xr.open_dataarray(filepath,engine='h5netcdf').load()
    name = da.name
    vals = da.values
    nnan = int(np.isnan(vals).sum())
    ntot = vals.size
    pct  = 100.0*nnan/ntot if ntot>0 else 0.0
    dims = ','.join(da.dims)
    print(f'{name:>14s}  {str(vals.shape):>30s}  {nnan:>12,}  {pct:>7.3f}%  {dims}')
    ALLVARS[name] = da
    if nnan>0:
        NANVARS[name] = da

print(f'\nVariables with NaN: {list(NANVARS.keys()) if NANVARS else "NONE"}')

In [ ]:
pspath = os.path.join(RAWDIR,'ERA5_surface_pressure.nc')
ps = xr.open_dataarray(pspath,engine='h5netcdf').load()
psvals = ps.values
nnan = int(np.isnan(psvals).sum())
print(f'Raw ps shape: {psvals.shape}  dims: {ps.dims}')
print(f'Raw ps NaN count: {nnan:,} ({100*nnan/psvals.size:.3f}%)')
print(f'Raw ps finite range: [{np.nanmin(psvals):.1f}, {np.nanmax(psvals):.1f}] hPa')

if nnan>0:
    nanmask = np.isnan(psvals)
    if 'time' in ps.dims:
        timedim = list(ps.dims).index('time')
        nanfrac_spatial = nanmask.mean(axis=timedim)
        print(f'\nSpatial NaN fraction range: [{nanfrac_spatial.min():.3f}, {nanfrac_spatial.max():.3f}]')
        print(f'Grid points with any NaN: {(nanfrac_spatial>0).sum()}')
        print(f'Grid points always NaN: {(nanfrac_spatial==1).sum()}')
else:
    print('\nNo NaN in raw ps.')

In [ ]:
if NANVARS:
    ncols = min(len(NANVARS),3)
    nrows = (len(NANVARS)+ncols-1)//ncols
    fig,axs = pplt.subplots(nrows=nrows,ncols=ncols,refwidth=2.8,proj='cyl',proj_kw={'lon_0':75})
    axs.format(coast=True,lonlim=LONRANGE,latlim=LATRANGE,
               lonlocator=10,latlocator=5,grid=False,
               suptitle='Time-Mean NaN Fraction by Variable')
    for i,(name,da) in enumerate(NANVARS.items()):
        ax = axs.flat[i]
        nanfrac = np.isnan(da.values).astype(float)
        if 'sig' in da.dims:
            nanfrac = nanfrac.max(axis=list(da.dims).index('sig'))
        if 'time' in da.dims:
            timedim = list(d for d in da.dims if d not in ('sig',)).index('time')
            nanfrac = nanfrac.mean(axis=timedim)
        lat = da.lat.values if 'lat' in da.coords else ALLVARS['tp'].lat.values
        lon = da.lon.values if 'lon' in da.coords else ALLVARS['tp'].lon.values
        m = ax.pcolormesh(lon,lat,nanfrac,cmap='Reds',vmin=0,vmax=1)
        ax.set_title(name)
    for j in range(i+1,nrows*ncols):
        axs.flat[j].axis('off')
    fig.colorbar(m,loc='b',label='NaN Fraction')
    pplt.show()
else:
    print('No variables with NaN — no map needed.')

In [ ]:
if NANVARS:
    sigvars = [name for name,da in NANVARS.items() if 'sig' in da.dims]
    if sigvars:
        print('NaN fraction by sigma level:\n')
        da0 = NANVARS[sigvars[0]]
        sigs = da0.sig.values
        print(f'{"sig":>6s}',end='')
        for name in sigvars:
            print(f'  {name:>14s}',end='')
        print()
        for si,s in enumerate(sigs):
            print(f'{s:6.2f}',end='')
            for name in sigvars:
                da = NANVARS[name]
                slc = da.isel(sig=si).values
                frac = np.isnan(slc).mean()
                print(f'  {frac:14.4%}',end='')
            print()
        print()
        print('If NaN fraction is identical across all sigma levels for a variable,')
        print('the NaN comes from invalid ps (entire column masked), not below-surface levels.')
else:
    print('No NaN in any interim variable.')

In [ ]:
if NANVARS:
    sigvars = [name for name,da in NANVARS.items() if 'sig' in da.dims]
    nonsigvars = [name for name in NANVARS if name not in sigvars]
    if sigvars:
        da0 = NANVARS[sigvars[0]]
        nanany = np.isnan(da0.values).any(axis=list(da0.dims).index('sig'))
        for name in sigvars[1:]:
            da = NANVARS[name]
            nanany_i = np.isnan(da.values).any(axis=list(da.dims).index('sig'))
            print(f'NaN pattern identical between {sigvars[0]} and {name}: {np.array_equal(nanany,nanany_i)}')
        print(f'\nTotal (time,lat,lon) points with NaN in sigma vars: {nanany.sum():,} / {nanany.size:,} ({100*nanany.mean():.3f}%)')
    if nonsigvars:
        print(f'\nNon-sigma variables with NaN: {nonsigvars}')
else:
    print('No NaN in any interim variable.')

In [ ]:
def kernel_integrate(fields,weights,dsig,mask=None):
    w = fields*weights[None,:,:]*dsig[None,None,:]
    if mask is not None: w *= mask[:,None,:]
    return w.sum(axis=2)

with xr.open_dataset(os.path.join(SPLITSDIR,'norm_train.h5'),engine='h5netcdf') as ds:
    ntime,nlat,nlon = ds.time.size,ds.lat.size,ds.lon.size
    nsig  = ds.sizes.get('sig',1)
    dsig  = ds.dsig.values
    fields = np.stack([ds[v].transpose('time','lat','lon','sig').values.reshape(-1,nsig) for v in FIELDVARS],axis=1)
    surfmask = ds.surfmask.transpose('time','lat','lon','sig').values.reshape(-1,nsig) if 'surfmask' in ds else None

print(f'surfmask present in splits: {surfmask is not None}')
print(f'Fields shape: {fields.shape}')
print(f'NaN in raw fields: {np.isnan(fields).sum():,} / {fields.size:,} ({100*np.isnan(fields).mean():.3f}%)')

kernels = [xr.open_dataset(os.path.join(WEIGHTSDIR,f'nn_gauss_{s}_weights.nc'),engine='h5netcdf')['k'].values for s in SEEDS]
meankernel = np.mean(kernels,axis=0)
print(f'NaN in kernel weights: {np.isnan(meankernel).sum()}')

integ = kernel_integrate(fields,meankernel,dsig,surfmask)
print(f'\nKernel-integrated features shape: {integ.shape}')
for i,v in enumerate(FIELDVARS):
    col = integ[:,i]
    nnan = np.isnan(col).sum()
    print(f'  {v}: NaN={nnan:,} / {len(col):,} ({100*nnan/len(col):.3f}%)',end='')
    if nnan>0:
        print(f'  finite range=[{np.nanmin(col):.4f}, {np.nanmax(col):.4f}]')
    else:
        print(f'  range=[{col.min():.4f}, {col.max():.4f}]')

nanpoints = ~np.isfinite(integ).all(axis=1)
print(f'\nPoints where ANY integrated fieldvar is NaN: {nanpoints.sum():,} / {len(nanpoints):,} ({100*nanpoints.mean():.3f}%)')

In [ ]:
if nanpoints.any():
    nanmap = nanpoints.reshape(ntime,nlat,nlon).mean(axis=0)
    with xr.open_dataset(os.path.join(SPLITSDIR,'norm_train.h5'),engine='h5netcdf') as ds:
        lat,lon = ds.lat.values,ds.lon.values

    fig,axs = pplt.subplots(ncols=1,refwidth=4,proj='cyl',proj_kw={'lon_0':75})
    axs.format(coast=True,lonlim=LONRANGE,latlim=LATRANGE,
               lonlocator=10,latlocator=5,grid=False,
               title='Fraction of Timesteps with NaN in Kernel-Integrated Features (Train)')
    m = axs.pcolormesh(lon,lat,nanmap,cmap='Reds',vmin=0,vmax=max(nanmap.max(),0.01))
    axs.colorbar(m,loc='b',label='NaN Fraction')
    pplt.show()

    print(f'Grid cells always NaN: {(nanmap==1).sum()}')
    print(f'Grid cells sometimes NaN: {((nanmap>0)&(nanmap<1)).sum()}')
    print(f'Grid cells never NaN: {(nanmap==0).sum()}')
else:
    print('No NaN in kernel-integrated features — all (time,lat,lon) points are valid.')